# 3. Two actors, one handle

An actor can hand something to another actor from inside a handler. `queue` puts it in the other
actor's inbox and returns a `Handle`. If the handler returns that handle, the calling command
settles when the other actor answers: its alias resolves to what the other actor emitted with
`OUT{}`, and the asking run's next turn starts with the answer in view. There is no team object
and no awaited tool call; there are two actors, two runs and one handle (ADR-0010).

Here a reviewer escalates a question to a researcher. Both are scripted, so the whole exchange is
on the page. It is stored with its outputs cleared; run it to see what each cell prints.

In [ ]:
import asyncio
import tempfile
from pathlib import Path

from pydantic import BaseModel, Field

from lion1 import Actor, Context, Kind, Profile, Run, RunRequest, Store
from lion1.khive import Khive

scratch = Path(tempfile.mkdtemp())  # the actor the runs' rows are written as
(scratch / ".khive").mkdir()
(scratch / ".khive" / "config.toml").write_text(
    # the actor, and the lion pack's tables routed to their own file beside a main store: the
    # store refuses a config that would put them in the default one under the home directory
    '[actor]\nid = "notebook"\n'
    f'\n[[backends]]\nname = "main"\npath = "{scratch / ".khive" / "main.db"}"\n'
    f'\n[[backends]]\nname = "lion"\npath = "{scratch / ".khive" / "lion.db"}"\n'
    '\n[packs.lion]\nbackend = "lion"\n'
)
store = Store(Khive(scratch, "notebook"))

In [ ]:
def show(run, width=140):
    """The record, one entry per line: what came in, what the system and the model said, what settled."""
    print(f"run {run.id}: {type(run.outcome).__name__} after {run.turn} turns\n")
    for e in run.record:
        if e.kind is Kind.RESULT:
            res = e.content
            body = f"{e.alias} <- {res.spec}: " + (res.preview(width) if res.ok else "ERR " + str(res.error))
        else:
            body = str(e.content)
        print(f"[{e.seq:>2} {e.name:>5} t{e.turn}] {body}\n")


def scripted(*turns):
    """A backend whose turns are written in advance: the view comes in, the next string goes out."""
    it = iter(turns)

    async def backend(messages):
        return next(it)

    return backend

In [ ]:
class Escalation(BaseModel):
    """Ask the researcher one precise question."""

    question: str
    severity: int = Field(ge=1, le=10)


class Finding(BaseModel):
    """The researcher's answer."""

    answer: str
    evidence: list[str]


class Verdict(BaseModel):
    """The reviewer's decision."""

    decision: str
    reason: str


researcher = Actor("researcher", store=store)
reviewer = Actor("reviewer", privileges={"escalate"}, store=store)
RESEARCHER = Profile("researcher", system="Answer the question you were handed; finish with OUT{}.")
REVIEWER = Profile(
    "reviewer", privileges=frozenset({"escalate"}), system="Review; escalate what you cannot settle."
)

researcher_turns = scripted(
    "<lvar finding.answer a1>version is read in notes.py get() and compared in tests/test_notes.py</lvar>\n"
    '<lvar finding.evidence e1>["src/lion1/notes.py", "tests/test_notes.py"]</lvar>\n'
    "OUT{finding: [a1, e1]}",
)
peer_runs: dict[str, Run] = {}
peer_tasks: set[asyncio.Task] = set()


@reviewer.handler(Escalation, requires={"escalate"})
async def escalate(req: Escalation, ctx: Context):
    if req.severity < 4:
        return "not escalated: severity below 4, decide it yourself"
    handle = researcher.queue(req, ctx)

    async def serve():
        peer_runs[handle.id] = await researcher.run(
            RESEARCHER, RunRequest(emits=(Finding,), max_turns=3), researcher_turns
        )

    peer_tasks.add(asyncio.create_task(serve()))  # held here, awaited below: its error is ours
    return handle

The reviewer's first turn escalates. The handler queues the question, starts the researcher's
run, and returns the handle; the command `q1` is now waiting on the answer. The researcher's
run begins with the escalation as its input, answers in one turn, and the answer settles `q1`.
The reviewer's second notification shows `q1` settled with the finding.

In [ ]:
run = await reviewer.run(
    REVIEWER,
    RunRequest(
        inputs=("Is the change to `version` in notes.py safe to merge?",), emits=(Verdict,), max_turns=4
    ),
    scripted(
        '<lact q1>escalation(question="Where is `version` read outside notes.py?", severity=7)</lact>',
        "<lvar verdict.decision d1>request_changes</lvar>\n"
        "<lvar verdict.reason r1>version is compared in tests; pinning it to 1 breaks them</lvar>\n"
        "OUT{verdict: [d1, r1]}",
    ),
)
show(run)

In [ ]:
await asyncio.gather(*peer_tasks)  # a peer that raised raises here, not as a warning at exit
for handle, peer in peer_runs.items():
    print(f"=== researcher run for handle {handle}")
    show(peer)

In [ ]:
print("q1 resolves to:", run.resolve("q1"))
print("verdict:", run.output["verdict"].model_dump())

Two things carry the design. The reviewer never saw the repository; its only path to a fact was
one Spec whose handler decided whether to ask at all (severity below 4 is answered locally).
And the answer arrived the same way every other result does: named, previewed in the
notification, resolvable by pointer. A run that ends without `OUT{}` settles the ask as a failed
result naming the outcome, and a run whose backend raises settles it naming the error, so the
asker reads why and goes on instead of waiting forever. Three tests in `tests/test_actor.py` pin
those paths; ADR-0010 is the record.